# IARA — Pesos Vivos IMAGEM (Colab)

A IARA **herda** o córtex de imagem (Stable Diffusion) e **aprende nos pesos** com feedback (LoRA) —
não frozen+seleção. Teste: ensinar um **conceito** que o SD base não sabe gerar e provar *closed-book*
que a adaptada gera onde a frozen não consegue. (Análogo imagético do córtex de linguagem.)

**Antes de rodar:** `Ambiente de execução → Alterar tipo de ambiente → GPU (T4)`. Depois `Ambiente → Executar tudo`.


## 1) Setup (GPU + libs)


In [ ]:
import torch
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução > Alterar tipo > GPU"
print("GPU:", torch.cuda.get_device_name(0))
!pip -q install -U "diffusers>=0.31" "peft>=0.11" accelerate
# peft novo exige torchao>0.16 (Colab tem 0.10). A LoRA não precisa dele — removemos p/ pular o check:
!pip -q uninstall -y torchao
print("setup ok — se instalou/removeu algo, use Ambiente de execução > Reiniciar sessão e Executar tudo")


## 2) O conceito que a IARA vai aprender

Uma textura procedural (listras azul/laranja + bolhas amarelas) que o SD base NÃO associa a nenhum prompt.


In [ ]:
import numpy as np, math
import matplotlib.pyplot as plt
from PIL import Image
PAL = np.array([[.30,.42,.72],[.72,.52,.28],[.95,.85,.10]], np.float32)  # azul, laranja, amarelo
def concept(seed, R=512):
    rng=np.random.default_rng(seed); y,x=np.mgrid[0:R,0:R].astype(np.float32)
    s=np.sign(np.sin((x*math.cos(.9)+y*math.sin(.9))*.05 + rng.uniform(0,6.28)))
    img=np.where(s[...,None]>0, PAL[0], PAL[1]).copy()
    for _ in range(14):
        cx,cy,r=rng.uniform(30,R-30),rng.uniform(30,R-30),rng.uniform(16,34)
        img[((x-cx)**2+(y-cy)**2)<r*r]=PAL[2]
    return img.astype(np.float32)
plt.figure(figsize=(3,3)); plt.imshow(concept(0)); plt.axis("off"); plt.title("o CONCEITO"); plt.show()


## 3) Herdar o córtex (SD-inpainting) + colar as sinapses (LoRA)


In [ ]:
import torch.nn.functional as F
from diffusers import StableDiffusionInpaintPipeline
from peft import LoraConfig
dev="cuda"
pipe=StableDiffusionInpaintPipeline.from_pretrained("runwayml/stable-diffusion-inpainting",
        torch_dtype=torch.float32, safety_checker=None).to(dev)
pipe.set_progress_bar_config(disable=True)
vae,unet,tok,txt,sched=pipe.vae,pipe.unet,pipe.tokenizer,pipe.text_encoder,pipe.scheduler
for m in (vae,txt,unet): m.requires_grad_(False)
unet.enable_gradient_checkpointing()
unet.add_adapter(LoraConfig(r=16,lora_alpha=16,init_lora_weights="gaussian",
        target_modules=["to_k","to_q","to_v","to_out.0"]))   # as SINAPSES da IARA sobre o córtex
lora=[p for p in unet.parameters() if p.requires_grad]
opt=torch.optim.AdamW(lora, lr=1e-4)
PROMPT="sks texture"; SF=vae.config.scaling_factor; R=512
with torch.no_grad():
    cond=txt(tok(PROMPT,padding="max_length",max_length=tok.model_max_length,return_tensors="pt").input_ids.to(dev))[0]
def encode(img):
    t=torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(dev,torch.float32)*2-1
    return vae.encode(t).latent_dist.sample()*SF
full_mask=torch.ones(1,1,R//8,R//8,device=dev)
with torch.no_grad(): masked_lat=encode(np.full((R,R,3),.5,np.float32))
print(f"LoRA: {sum(p.numel() for p in lora)/1e6:.2f}M treináveis · córtex CONGELADO")


## 4) Treinar (feedback = 'isto é o certo') — o loss deve CAIR


In [ ]:
STEPS=700; losses=[]
for step in range(STEPS):
    with torch.no_grad():
        lat=encode(concept(1000+step%10))          # 10 instâncias do conceito
        noise=torch.randn_like(lat)
        t=torch.randint(0,sched.config.num_train_timesteps,(1,),device=dev).long()
        model_in=torch.cat([sched.add_noise(lat,noise,t), full_mask, masked_lat],1)
    loss=F.mse_loss(unet(model_in,t,encoder_hidden_states=cond).sample, noise)
    loss.backward(); opt.step(); opt.zero_grad(); losses.append(loss.item())
    if step%50==0: print(f"passo {step:4d} · loss(m.móvel) {np.mean(losses[-25:]):.4f}")
plt.figure(figsize=(6,3)); plt.plot(np.convolve(losses,np.ones(25)/25,"valid"))
plt.title("loss (média-móvel) — cair = aprendeu"); plt.show()


## 5) Teste closed-book: GERAR o conceito do zero — frozen vs IARA-adaptada


In [ ]:
def gen(use_lora, seed):
    unet.enable_adapters() if use_lora else unet.disable_adapters()
    blank=Image.new("RGB",(R,R),(128,128,128)); allm=Image.new("L",(R,R),255)
    with torch.no_grad():
        out=pipe(prompt=PROMPT,image=blank,mask_image=allm,num_inference_steps=30,guidance_scale=7.0,
                 generator=torch.Generator(dev).manual_seed(seed)).images[0].resize((R,R))
    return np.asarray(out,np.float32)/255
def palette(img):
    d=np.min([np.abs(img-c).sum(-1) for c in PAL],0); return float((d<0.25).mean())
fig,ax=plt.subplots(1,4,figsize=(16,4))
ax[0].imshow(concept(50)); ax[0].set_title("conceito-alvo")
fr=gen(False,0); ax[1].imshow(np.clip(fr,0,1)); ax[1].set_title(f"FROZEN · paleta {palette(fr):.0%}")
a0=gen(True,0);  ax[2].imshow(np.clip(a0,0,1)); ax[2].set_title(f"IARA-adaptada · {palette(a0):.0%}")
a2=gen(True,2);  ax[3].imshow(np.clip(a2,0,1)); ax[3].set_title(f"IARA seed2 · {palette(a2):.0%}")
for a in ax: a.axis("off")
plt.show()
print("FROZEN gera ~0% do conceito; ADAPTADA gera >> 0% = aprendeu criação de imagem NOS PESOS.")


## 6) Pra ir além (é aqui que o Colab paga)
- **Estabilizar**: sobe `STEPS` (1500+) e/ou `r=32` → o recall vai de ~43% pra consistente.
- **Escalar**: ensina 2 conceitos com tokens diferentes e vê se ela guarda os dois (sem esquecer).
- **Pro (A100)**: troca por **SDXL-inpainting** = geração muito melhor.
- **Próximo notebook**: IARA-Imagem nativa — solver-água NO LATENTE do VAE, pirâmide = passos (a arquitetura que desenhamos).
